# Week 3 – Advanced Data Analysis and Visualization in Logistics

**YuvaIntern – Logistics Data Analyst Internship**

This notebook performs exploratory data analysis (EDA) on the cleaned Week 2 logistics dataset. It calculates central tendencies, distributions, correlations, and operational metrics, then creates visualizations to identify delivery delays, cost patterns, regional bottlenecks, and potential operational risks.


## 1. Load the Week 2 cleaned dataset

The dataset is stored in the repository under `data/`. The notebook is stored under `notebooks/`, so the relative path is `../data/week2_cleaned_logistics_data.csv`.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("../data/week2_cleaned_logistics_data.csv")
df["Order_Date"] = pd.to_datetime(df["Order_Date"], errors="coerce")

# Standardize a minor label variation for analysis
df["Shipping_Mode"] = df["Shipping_Mode"].replace({"Same day": "Same Day"})
df["Late_Delivery_Flag"] = (df["Delay_Days"] > 0).astype(int)

print("Shape:", df.shape)
display(df.head())


## 2. Data overview and quality check

In [ ]:
print(df.info())
print("\nMissing values:")
display(df.isna().sum())

print("\nDuplicate rows:", df.duplicated().sum())


## 3. Central tendency and descriptive statistics

Mean and median help understand the typical logistics values. Standard deviation shows variability.


In [ ]:
numeric_cols = [
    "Shipment_Quantity", "Product_Price", "Shipping_Cost",
    "Scheduled_Shipping_Days", "Actual_Shipping_Days",
    "Customer_Rating", "Delay_Days"
]

summary = df[numeric_cols].agg(["count", "mean", "median", "std", "min", "max"]).T
display(summary.round(2))


## 4. Key logistics performance metrics

In [ ]:
late_rate = df["Late_Delivery_Flag"].mean() * 100
avg_delay = df["Delay_Days"].mean()
avg_shipping_cost = df["Shipping_Cost"].mean()
avg_actual_days = df["Actual_Shipping_Days"].mean()

print(f"Late delivery rate: {late_rate:.2f}%")
print(f"Average delay: {avg_delay:.2f} days")
print(f"Average shipping cost: {avg_shipping_cost:.2f}")
print(f"Average actual shipping time: {avg_actual_days:.2f} days")


## 5. Visualization 1 – Distribution of actual shipping days

**Why this chart?** A histogram shows how delivery times are distributed and whether most shipments cluster around a small number of days.


In [ ]:
plt.figure(figsize=(7, 4.5))
plt.hist(df["Actual_Shipping_Days"], bins=np.arange(
    df["Actual_Shipping_Days"].min()-0.5,
    df["Actual_Shipping_Days"].max()+1.5, 1
))
plt.axvline(df["Actual_Shipping_Days"].mean(), linestyle="--",
            label=f"Mean = {df['Actual_Shipping_Days'].mean():.2f}")
plt.title("Distribution of Actual Shipping Days")
plt.xlabel("Actual Shipping Days")
plt.ylabel("Number of Shipments")
plt.legend()
plt.tight_layout()
plt.show()


## 6. Visualization 2 – Average delay by shipping mode

**Why this chart?** A bar chart makes it easy to compare operational delay across transportation/service modes.


In [ ]:
mode_stats = df.groupby("Shipping_Mode").agg(
    Orders=("Order_ID", "count"),
    Avg_Delay_Days=("Delay_Days", "mean"),
    Late_Rate=("Late_Delivery_Flag", "mean"),
    Avg_Shipping_Cost=("Shipping_Cost", "mean")
)
mode_stats["Late_Rate"] *= 100
display(mode_stats.round(2))

plt.figure(figsize=(7, 4.5))
plt.bar(mode_stats.index, mode_stats["Avg_Delay_Days"])
plt.title("Average Delay by Shipping Mode")
plt.xlabel("Shipping Mode")
plt.ylabel("Average Delay (days)")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


## 7. Visualization 3 – Late delivery rate by region

**Why this chart?** Regional comparison helps identify geographic bottlenecks and locations where delivery reliability needs improvement.


In [ ]:
region_stats = df.groupby("Region").agg(
    Orders=("Order_ID", "count"),
    Avg_Delay_Days=("Delay_Days", "mean"),
    Late_Rate=("Late_Delivery_Flag", "mean"),
    Avg_Shipping_Cost=("Shipping_Cost", "mean")
)
region_stats["Late_Rate"] *= 100
display(region_stats.round(2))

plt.figure(figsize=(7, 4.5))
plt.bar(region_stats.index, region_stats["Late_Rate"])
plt.title("Late Delivery Rate by Region")
plt.xlabel("Region")
plt.ylabel("Late Deliveries (%)")
plt.tight_layout()
plt.show()


## 8. Visualization 4 – Shipping cost vs product price

**Why this chart?** A scatter plot shows whether higher-value shipments are associated with higher transportation costs and reveals unusual observations.


In [ ]:
corr_cost_price = df[["Product_Price", "Shipping_Cost"]].corr().iloc[0, 1]
print(f"Correlation between product price and shipping cost: {corr_cost_price:.2f}")

plt.figure(figsize=(7, 4.5))
plt.scatter(df["Product_Price"], df["Shipping_Cost"], alpha=0.65)
plt.title("Shipping Cost vs Product Price")
plt.xlabel("Product Price")
plt.ylabel("Shipping Cost")
plt.tight_layout()
plt.show()


## 9. Visualization 5 – Delay distribution by product category

**Why this chart?** A box plot compares the spread and central tendency of delivery delays across categories and can highlight categories with greater variability.


In [ ]:
categories = sorted(df["Product_Category"].unique())

plt.figure(figsize=(7, 4.5))
plt.boxplot(
    [df.loc[df["Product_Category"] == c, "Delay_Days"] for c in categories],
    labels=categories
)
plt.title("Delay Distribution by Product Category")
plt.xlabel("Product Category")
plt.ylabel("Delay (days)")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


## 10. Visualization 6 – Correlation matrix

**Why this chart?** A correlation matrix summarizes linear relationships among numerical logistics variables and helps prioritize variables for deeper analysis or future modeling.


In [ ]:
corr_cols = [
    "Shipment_Quantity", "Product_Price", "Shipping_Cost",
    "Scheduled_Shipping_Days", "Actual_Shipping_Days",
    "Customer_Rating", "Delay_Days"
]
corr = df[corr_cols].corr()

plt.figure(figsize=(8, 6))
plt.imshow(corr.values, aspect="auto")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=45, ha="right")
plt.yticks(range(len(corr.columns)), corr.columns)
for i in range(len(corr.columns)):
    for j in range(len(corr.columns)):
        plt.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
plt.title("Correlation Matrix of Key Logistics Variables")
plt.colorbar()
plt.tight_layout()
plt.show()


## 11. Analytical insights

1. The overall late-delivery rate is approximately 43.5%, indicating that delivery reliability is an important operational area to monitor.
2. Actual shipping days have a relatively compact distribution, but delays still occur because actual days can exceed scheduled days.
3. Regional comparison highlights differences in late-delivery rates; the highest-rate region should receive priority for bottleneck investigation.
4. Shipping mode comparison helps identify modes with relatively higher average delays and late rates.
5. Product category comparison helps identify categories with higher average delay or wider variability.
6. The correlation matrix indicates that actual shipping days have the strongest relationship with delay days, which is expected because delay is derived from actual versus scheduled shipping time.
7. The cost-price scatter plot helps examine whether higher-value products are associated with higher shipping costs and identifies observations for further investigation.

These findings can support operational decisions such as reviewing regional carrier performance, monitoring high-delay shipping modes, and investigating categories with persistent delays.


## 12. Conclusion

The EDA demonstrates how Python can convert a cleaned logistics dataset into actionable operational information. Descriptive statistics establish the baseline, while visualizations reveal patterns in delivery time, delay, cost, region, shipping mode, and product category. The analysis provides a foundation for Week 4 predictive modeling or advanced performance analysis.
